In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PolynomialFeatures
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier, VotingClassifier, BaggingClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix
import joblib

In [2]:
n = 1500

In [3]:
np.random.seed(20)

In [4]:
X = pd.DataFrame({
    'moisture': np.random.uniform(0, 20, n).round(2),
    'temp': np.random.uniform(0, 40, n).round(1),
    'sunlight': np.random.randint(0, 12, n),
    'fertilizer': np.random.randint(100, 400, n),
    'rainfall': np.random.randint(0, 50, n),
    'size': np.random.randint(500, 5000, n),
    'crop': np.random.choice(['wheat', 'corn', 'rice', 'cotton'], n),
    'irrigation': np.random.choice(['low', 'medium', 'high'], n),
})

In [5]:
y = (100
     - 0.3 * X['moisture']
     + 0.4 * X['temp']
     - 2 * X['rainfall']
     + 0.3 * X['sunlight']
     + X['irrigation'].map({'low': 2, 'medium': 5, 'high': 12})
     + X['crop'].map({'wheat': -2, 'corn': 3, 'cotton': 0, 'rice': 5})
     + np.random.uniform(-0.5, 0.5, n)
)

In [6]:
y2 = (y >= 80).astype(int)

In [7]:
X.loc[X.sample(frac = 0.1, random_state = 20).index, 'moisture'] = np.nan

In [8]:
outliers = X.sample(frac = 0.03, random_state = 10).index
outliers2 = X.sample(frac = 0.02, random_state = 30).index

In [9]:
X.loc[outliers, 'temp'] = np.random.choice([-40, 85], len(outliers))
X.loc[outliers2, 'size'] = np.nanmean(X['size']) * 20

C:\Users\Zcc\AppData\Local\Temp\ipykernel_9864\2170105508.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '54403.33333333333' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  X.loc[outliers2, 'size'] = np.nanmean(X['size']) * 20


In [10]:
noise = np.random.uniform(0, 2, n).round(2)

In [11]:
X['rainfall'] = X['rainfall'] + noise
X['fertilizer'] = X['fertilizer'] + noise

In [12]:
num_features = ['moisture', 'temp', 'sunlight', 'fertilizer', 'rainfall', 'size']
ord_features = ['irrigation'] 
nom_features = ['crop']

In [13]:
for col in ['temp', 'size']:
    Q1 = X[col].quantile(0.25)
    Q3 = X[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    X[col] = X[col].clip(lower, upper)

In [14]:
for col in num_features:
    X[col] = X[col].fillna(X[col].mean())

In [15]:
pp = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('ord', OrdinalEncoder(categories = [['low', 'medium', 'high']]), ord_features),
    ('nom', OneHotEncoder(handle_unknown = 'ignore'), nom_features)
])

In [16]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.15, random_state = 42)

In [17]:
x2_train, x2_test, y2_train, y2_test = train_test_split(X, y2, test_size = 0.15, random_state = 32)

In [18]:
lr = Pipeline([
    ('pp', pp),
    ('model', LinearRegression())
])

In [19]:
lr.fit(x_train, y_train)
lrp = lr.predict(x_test)

In [20]:
poly2 = Pipeline([
    ('pp', pp),
    ('poly', PolynomialFeatures(degree = 2)),
    ('model', LinearRegression())
])

In [21]:
poly2.fit(x_train, y_train)
poly2p = poly2.predict(x_test)

In [22]:
poly3 = Pipeline([
    ('pp', pp),
    ('poly', PolynomialFeatures(degree = 3)),
    ('model', LinearRegression())
])

In [23]:
poly3.fit(x_train, y_train)
poly3p = poly3.predict(x_test)

In [24]:
log = Pipeline([
    ('pp', pp),
    ('model', LogisticRegression(max_iter = 1000, random_state = 30))
])

In [25]:
log.fit(x2_train, y2_train)
logp = log.predict(x_test)

In [26]:
results = []

In [27]:
for k in [3, 5, 7, 9 ,11]:
    knn = Pipeline([
        ('pp', pp),
        ('model', KNeighborsClassifier(n_neighbors = int(k), metric = 'euclidean'))
    ])

    knn.fit(x2_train, y2_train)
    knnp = knn.predict(x2_test)

    knn_acc = accuracy_score(y2_test, knnp)

    results.append((k, knn_acc))

In [28]:
results_df = pd.DataFrame(
    results,
    columns = ['k', 'knn_acc']
)

In [29]:
best_knn = results_df.loc[results_df['knn_acc'].idxmax()]

In [30]:
knn_best = Pipeline([
    ('pp', pp),
    ('model', KNeighborsClassifier(n_neighbors = int(best_knn.k), metric = 'euclidean'))
])

In [31]:
knn_best.fit(x2_train, y2_train)
knn_bestp = knn_best.predict(x2_test)

In [32]:
lr_mse = mean_squared_error(y_test, lrp)
poly2_mse = mean_squared_error(y_test, poly2p)
poly3_mse = mean_squared_error(y_test, poly3p)

In [33]:
log_acc = accuracy_score(y2_test, logp)
knn_best_acc = accuracy_score(y2_test, knn_bestp)

In [34]:
models = {'lr': lr, 'poly2': poly2, 'poly3': poly3, 'log': log, 'knn': best_knn}
mses = {'lr': lr_mse, 'poly2': poly2_mse, 'poly3': poly3_mse}
accs = {'log': log_acc, 'knn': knn_best_acc}

In [35]:
best_reg_mse = min(mses)
best_cls_acc = max(accs)

In [36]:
best_reg_model = models[best_reg_mse]
best_cls_model = models[best_cls_acc]

In [37]:
joblib.dump(best_reg_model, 'regressor.joblib')
joblib.dump(best_cls_model, 'classifier.joblib')

['classifier.joblib']